# Análisis de distribución de precios

El objetivo de este notebook es analizar la distribución del precio de los juegos en Steam

---

## Importación de librerías

In [ ]:
# Módulos
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
# Para que se puedan utilizar funciones desde el notebook
import sys
from os import path
root_path = path.abspath(path.join('..', '..'))
if root_path not in sys.path:
    sys.path.append(root_path)
from src.utils.files import read_file
from src.utils.config import prices, load_env_file

load_env_file()

## Carga de datos
Configuración del Minio, poner en True para usar la información de Minio

In [ ]:
use_minio = True # Solo cambiar este parámetro
minio = {"minio_write": False, "minio_read": use_minio}

In [ ]:
df = read_file(prices, minio)
df.head(2)

---
## Análisis
En este DataFrame todos los juegos son no gratuitos, puesto que para el problema de predicción de precio sólo se tienen en cuenta juegos que no sean gratis.

### Pie Chart de rango de precios

In [ ]:
fig = go.Figure()

orden = [
    "[0.01,4.99]",
    "[5.00,9.99]",
    "[10.00,14.99]",
    "[15.00,19.99]",
    "[20.00,29.99]",
    "[30.00,39.99]",
    ">40"
]

values_df = df["price_range"].value_counts()

values_df = values_df.reindex(orden)

fig.add_trace(go.Pie(values= values_df.values, labels= values_df.index, sort = False))

fig.update_layout(
                  paper_bgcolor = "#131416",
                  width = 800,
                  title = {
                      "text": "Proporción de los rangos de precio",
                      "x": 0.5,
                      "y":0.9,
                      "font":dict(family="Verdana", size = 20, weight="bold")
                  },
                  font_color = "white")

fig.show()

Casi el 50% de los juegos no gratuitos tienen un precio inferior a 5€.

### Histograma de precios con transformación logarítmica

In [ ]:
fig = go.Figure()

fig.add_trace(go.Histogram(x = df['price_overview'],
                           marker_color = "#99AA36",
                           xbins = dict(start=df['price_overview'].min(), end = df['price_overview'].max() + 5,
                                        size = 5)))

fig.update_layout(
    paper_bgcolor = "#131416",
    plot_bgcolor = "#F0F7CC",
    
    xaxis_title = "Precio (€)",
    yaxis_title = "Número de juegos (log)",
    yaxis = dict(type = "log"),
    xaxis = dict(range = [0,150], tickmode = "linear", dtick = 5),
    
    xaxis_ticks = "outside",
    yaxis_ticks = "outside",
    xaxis_linecolor = "white",
    yaxis_linecolor = "white",
    
    font_color = "white",
    title = {
            "text": "Distribución de los precios",
            "x": 0.5,
            "y":0.9,
            "font":dict(family="Verdana", size = 20, weight="bold")
            }
    
)
              

fig.show()

- En el histograma se han retirado aquellos juegos con un precio superior a los 150€, pues son considerados "outliers" y no ayudan a identificar la distribución de los datos. Además estos juegos no suelen ser juegos serios sino que son juegos "cutres" lanzados por precios ingentes que no tienen justificación alguna.

- Es importante mencionar que el eje y tiene escala logarítmica. A medida que aumenta el precio de los juegos, el número de estos en cada precio se reduce drásticamente.

- Esto último es coherente con el hecho de que en Steam gran parte de los juegos pertencen al género "Indie" (juegos independientes hechos por un grupo pequeño), donde los precios rara vez superan los 20-30€, mientras que en el otro extremo se encuentran los juegos "AAA" que pueden llegar a alcanzar los 80€ o incluso más por sus altos costes de producción.

### Deshacer MultilabelBinarizer

In [ ]:
genres = ['Action', 'Adventure', 'Casual', 'Early Access', 'Indie', 
    'RPG', 'Simulation', 'Strategy', 'Co-op']

mlb = MultiLabelBinarizer()
mlb.fit([genres])
inverse_transformation = mlb.inverse_transform(df[genres].values)
# df.drop(columns=genres, inplace=True)
df['genres'] = inverse_transformation
df.head(2)


### Pie Chart de la distribución de los géneros de cada rango de precios.

In [ ]:
fig = make_subplots(
    rows=2, cols=4,
    specs=[
        [{"type": "domain"}, {"type": "domain"}, {"type": "domain"}, {"type": "domain"}],
        [{"type": "domain"}, {"type": "domain"}, {"type": "domain"}, {"type": "domain"}]
    ],
    subplot_titles=[
        "[0.01,4.99]", "[5.00,9.99]", "[10.00,14.99]",
        "[15.00,19.99]", "[20.00,29.99]", "[30.00,39.99]", ">40"
    ]
)

df_exploded = df.explode('genres')

price_genre_counts = df_exploded.groupby(['price_range', 'genres']).size().unstack(fill_value=0)

orden = {
    "[0.01,4.99]": [1,1],
    "[5.00,9.99]": [1,2],
    "[10.00,14.99]": [1,3],
    "[15.00,19.99]": [1,4],
    "[20.00,29.99]": [2,1],
    "[30.00,39.99]": [2,2],
    ">40": [2,3]
}

# Número máximo de géneros a mostrar por rango de precio
N = 5

for rango_precio, pos in orden.items():
    genre_counts = price_genre_counts.loc[rango_precio]
    
    # Filtramos solo los géneros con más juegos
    genre_counts = genre_counts.sort_values(ascending=False).head(N)
    
    # Solo agregamos si hay al menos un género
    if not genre_counts.empty:
        fig.add_trace(
            go.Pie(
                values=genre_counts.values,
                labels=genre_counts.index,
                sort=False,
                name = rango_precio,
                
                textposition= "inside"
            ),
            row=pos[0], col=pos[1]
        )
    
fig.update_layout(
    paper_bgcolor="#131416",
    font_color="white",
    title={
        "text": "Distribución de los géneros por cada rango de precio",
        "x": 0.5,
        "y": 0.93,
        "font": dict(family="Verdana", size=20, weight="bold")
    },
    legend_title = "Géneros",
    legend = dict(
        x = 0.87,
        y = -0.1,
        font = dict(
            size = 10)
    )
)

fig.show()

- Se puede apreciar que los porcentajes de los géneros más populares de cada rango varían significativamente. 

- El ejemplo más claro sería el género "Indie". Como se mencionó anteriormente, los juegos de este género abundan en los rangos de precio inferiores y
a medida que aumenta el precio su porcentaje en cada rango disminuye.

- Otro ejemplo bastante representativo sería el del género "Casual". Los juegos de este género suelen ser juegos sencillos y no muy largos, haciéndolos accesibles a la mayor cantidad de público posible. Como resultado, muchos de estos juegos se encuentran relacionados con los juegos "Indie", y están más alejados de juegos tipo "Estrategia" o "Acción" que suelen abundar en las grandes producciones (juegos "AAA"). Es por ello que el porcentaje de juegos "Casual" disminuye a medida que aumenta el precio.

### Bar plot de los precios y los géneros

In [ ]:
genre_price_counts = df_exploded.groupby(['genres', 'price_range']).size().unstack(fill_value=0)

totales = genre_price_counts.sum(axis=1)

generos_ordenados = totales.sort_values(ascending=True).index

# Se toman los 10 géneros más populares
generos_ordenados = generos_ordenados[-10:]

orden_precios = ["[0.01,4.99]", "[5.00,9.99]", "[10.00,14.99]",
                 "[15.00,19.99]", "[20.00,29.99]", "[30.00,39.99]", ">40"]

fig = go.Figure()

for price in orden_precios:
    if price in genre_price_counts.columns:
        fig.add_trace(
            go.Bar(
                y=generos_ordenados,                  # aquí usamos el orden total
                x=genre_price_counts.loc[generos_ordenados, price],
                name=price,
                orientation='h'
            )
        )

fig.update_layout(
    paper_bgcolor = "#131416",
    plot_bgcolor = "#ECE8E8",
    
    font_color = "white",
    barmode='stack',
    
    title={"text":'Distribución de precio por género (orden total)',
           "x":0.5,
           "y":0.95,
           "font": dict(family="Verdana", size=20, weight="bold")
           },
    xaxis_title='Número de juegos',
    yaxis_title='Género',
    
    xaxis_ticks = "outside",
    yaxis_ticks = "outside",
    
    xaxis_linecolor = "white",
    yaxis_linecolor = "white",
    
    legend_title='Precio',
    height=600,
    width=900
)

fig.show()

Dentro de cada género las proporciones son similares a las observadas en el primer gráfico (proporción de rangos de precio).

- - -

## Conclusión

- La distribución es una cola larga, con una gran concentración de juegos en los precios bajos (en gran parte debido a la cantidad de juegos "Indie"), y una minoría superando la barrera de los juegos "AAA" (60€ o más).

- Es importante notar que dentro de cada rango de precio las proporciones de géneros varían significativamente y de manera coherente.

- Hay géneros que se mantienen bastante constantes a lo largo de los distintos rangos de precio (ej: "Aventura"), por otro lado existen géneros que se localizan principalmente en rangos específicos (ej: "Indie", "Casual").